In [5]:
#!/usr/bin/env python3
"""
Part D — Complete pipeline
- ONNX export
- TensorRT FP32 / FP16 / INT8 (train-subset calibrator)
- PyTorch CPU/GPU benchmarking
- Batch scaling study (1, batch_size, 32)
- Plots + Markdown + PDF report
- Dual-mode (Jupyter / CLI)
"""

import os
import time
import argparse
import json
import math
from pathlib import Path
from typing import Dict

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Optional (TensorRT / PyCUDA)
TRT_AVAILABLE, PYCUDA_AVAILABLE = False, False
try:
    import tensorrt as trt
    TRT_AVAILABLE = True
except Exception:
    trt = None
try:
    import pycuda.driver as cuda
    import pycuda.autoinit
    PYCUDA_AVAILABLE = True
except Exception:
    cuda = None

# Defaults (change if needed)
DEFAULT_DATA_DIR = "/home/nagaraj/Garbage_classification_files/Garbage classification"
DEFAULT_CHECKPOINT = "partB_results/best_model.pth"
RESULTS_DIR = "partD_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------
# Model (same architecture used in A/B)
# ---------------------------
class FlexibleCNNSimple(nn.Module):
    def __init__(self, input_size=224, num_classes=6,
                 filters=(32,64,128,256,512), kernel_sizes=(3,3,3,3,3),
                 activation="relu", dense_neurons=512, use_bn=True, dropout=0.3):
        super().__init__()
        def get_act(name):
            if name == "relu": return nn.ReLU(inplace=False)
            if name == "gelu": return nn.GELU()
            if name in ("silu","swish"): return nn.SiLU()
            if name == "mish": return nn.Mish()
            return nn.ReLU(inplace=False)
        act = get_act(activation)
        layers = []
        in_ch = 3
        for i in range(len(filters)):
            layers.append(nn.Conv2d(in_ch, filters[i], kernel_size=kernel_sizes[i], padding=kernel_sizes[i]//2))
            if use_bn:
                layers.append(nn.BatchNorm2d(filters[i]))
            layers.append(act)
            layers.append(nn.MaxPool2d(2))
            in_ch = filters[i]
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(filters[-1], dense_neurons)
        self.act = act
        self.dropout = nn.Dropout(dropout) if dropout>0 else None
        self.fc2 = nn.Linear(dense_neurons, num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.act(x)
        if self.dropout:
            x = self.dropout(x)
        x = self.fc2(x)
        return x

# ---------------------------
# Utilities: ONNX export
# ---------------------------
def export_onnx(model: nn.Module, input_size: int, onnx_path: str):
    model.eval()
    dummy = torch.randn(1, 3, input_size, input_size, device="cpu")
    torch.onnx.export(
        model.cpu(), dummy, onnx_path, opset_version=13, verbose=False,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
    )
    print(f"[OK] ONNX exported to: {onnx_path}")

# ---------------------------
# Calibration loader: use a small RANDOM subset of TRAIN set
# ---------------------------
def get_calibration_loader(data_dir: str, input_size: int, batch_size: int = 8, num_samples: int = 200):
    train_dir = os.path.join(data_dir, "train")
    if not os.path.isdir(train_dir):
        raise RuntimeError(f"Train dir not found for calibration: {train_dir}")
    tf = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    ds = datasets.ImageFolder(train_dir, transform=tf)
    n = min(len(ds), num_samples)
    if n == 0:
        raise RuntimeError("No images found in train folder for calibration.")
    idx = np.random.choice(len(ds), size=n, replace=False).tolist()
    sub = Subset(ds, idx)
    loader = DataLoader(sub, batch_size=batch_size, shuffle=True, num_workers=0)
    return loader

# ---------------------------
# PyTorch evaluation (accuracy, latency, throughput)
# ---------------------------
def evaluate_pytorch(model: nn.Module, loader: DataLoader, device: torch.device):
    model = model.to(device)
    model.eval()
    correct, total = 0, 0
    times = []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            t0 = time.perf_counter()
            out = model(imgs)
            t1 = time.perf_counter()
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            times.append((t1 - t0) / imgs.size(0))
    acc = correct / total if total > 0 else 0.0
    latency_ms = float(np.mean(times) * 1000.0) if times else float("nan")
    throughput = float(total / sum(times)) if times and sum(times) > 0 else float("nan")
    return {"accuracy": acc, "latency_ms": latency_ms, "throughput_img_s": throughput, "batch_size": loader.batch_size}

# ---------------------------
# TensorRT: calibrator and engine build
# ---------------------------
if TRT_AVAILABLE:
    TRT_LOGGER = trt.Logger(trt.Logger.ERROR)

    class EntropyCalibrator(trt.IInt8EntropyCalibrator2):
        def __init__(self, calib_loader, cache_file="calib_cache.bin", device_input_ptr=None, max_batches=10):
            super().__init__()
            self.loader = calib_loader
            self.iterator = iter(self.loader)
            self.cache_file = cache_file
            self.max_batches = max_batches
            self.current = 0
            self.device_input = None

        def get_batch_size(self):
            return self.loader.batch_size

        def get_batch(self, names):
            if self.current >= self.max_batches:
                return None
            try:
                imgs, _ = next(self.iterator)
            except StopIteration:
                return None
            arr = imgs.numpy().astype(np.float32)
            if self.device_input is None:
                import pycuda.driver as cuda
                self.device_input = cuda.mem_alloc(arr.nbytes)
            cuda.memcpy_htod(self.device_input, arr)
            self.current += 1
            return [int(self.device_input)]

        def read_calibration_cache(self):
            if os.path.exists(self.cache_file):
                with open(self.cache_file, "rb") as f:
                    return f.read()
            return None

        def write_calibration_cache(self, cache):
            with open(self.cache_file, "wb") as f:
                f.write(cache)

    def build_trt_engine_from_onnx(onnx_path: str, fp16: bool=False, int8: bool=False, calibrator=None, workspace_size=(1<<30)):
        builder = trt.Builder(TRT_LOGGER)
        explicit_batch = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        network = builder.create_network(explicit_batch)
        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError("Failed to parse ONNX")
        config = builder.create_builder_config()
        config.max_workspace_size = workspace_size
        if fp16:
            config.set_flag(trt.BuilderFlag.FP16)
        if int8:
            config.set_flag(trt.BuilderFlag.INT8)
            if calibrator is None:
                raise RuntimeError("INT8 requested but no calibrator provided")
            config.int8_calibrator = calibrator
        engine = builder.build_engine(network, config)
        if engine is None:
            raise RuntimeError("Engine build failed")
        return engine

# ---------------------------
# TensorRT inference + evaluation (requires PyCUDA)
# ---------------------------
def evaluate_trt_engine(engine, loader: DataLoader):
    if not (TRT_AVAILABLE and PYCUDA_AVAILABLE):
        return {"error": "TensorRT or PyCUDA not available"}
    import pycuda.driver as cuda
    context = engine.create_execution_context()
    # assuming binding 0 = input, 1 = output; shapes may include -1 for dynamic dim
    in_binding = 0
    out_binding = 1
    in_shape = engine.get_binding_shape(in_binding)
    out_shape = engine.get_binding_shape(out_binding)
    dtype = np.float32
    # allocate device buffers
    d_input = cuda.mem_alloc(int(trt.volume(in_shape) * np.dtype(dtype).itemsize))
    d_output = cuda.mem_alloc(int(trt.volume(out_shape) * np.dtype(dtype).itemsize))
    stream = cuda.Stream()
    correct, total = 0, 0
    times = []
    for imgs, labels in loader:
        imgs_np = imgs.numpy().astype(np.float32)
        cuda.memcpy_htod_async(d_input, imgs_np, stream)
        t0 = time.perf_counter()
        context.execute_v2([int(d_input), int(d_output)])
        stream.synchronize()
        t1 = time.perf_counter()
        # read output
        out_np = np.empty(trt.volume(out_shape), dtype=np.float32)
        cuda.memcpy_dtoh_async(out_np, d_output, stream)
        stream.synchronize()
        # reshape output to (batch, classes)
        out_np = out_np.reshape((loader.batch_size, -1))
        preds = out_np.argmax(axis=1)
        correct += (preds == labels.numpy()).sum()
        total += labels.size(0)
        times.append((t1 - t0) / imgs.shape[0])
    acc = float(correct) / float(total) if total > 0 else 0.0
    latency_ms = float(np.mean(times) * 1000.0) if times else float("nan")
    throughput = float(total / sum(times)) if times and sum(times) > 0 else float("nan")
    return {"accuracy": acc, "latency_ms": latency_ms, "throughput_img_s": throughput, "batch_size": loader.batch_size}

# ---------------------------
# Plots: accuracy vs latency and throughput bar chart
# ---------------------------
def plot_accuracy_latency(results: Dict, out_path: str):
    labels = []
    accs = []
    lats = []
    for k, v in results.items():
        if isinstance(v, dict) and "accuracy" in v:
            labels.append(k)
            accs.append(v["accuracy"])
            lats.append(v["latency_ms"])
    if not labels:
        return None
    fig, ax1 = plt.subplots(figsize=(12,5))
    ax2 = ax1.twinx()
    ax1.bar(labels, accs, color="tab:blue", alpha=0.7, label="Accuracy")
    ax2.plot(labels, lats, color="tab:red", marker="o", label="Latency (ms/img)")
    ax1.set_ylabel("Accuracy")
    ax2.set_ylabel("Latency (ms/img)")
    plt.xticks(rotation=45, ha="right")
    plt.title("Accuracy (bar) vs Latency (line) per backend")
    fig.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)
    return out_path

def plot_throughput(results: Dict, out_path: str):
    labels = []
    thr = []
    for k, v in results.items():
        if isinstance(v, dict) and "throughput_img_s" in v:
            labels.append(k)
            thr.append(v["throughput_img_s"])
    if not labels:
        return None
    fig, ax = plt.subplots(figsize=(10, max(4, len(labels)*0.5)))
    ax.barh(labels, thr, color="tab:green")
    ax.set_xlabel("Throughput (images/sec)")
    plt.title("Throughput comparison")
    plt.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)
    return out_path

# ---------------------------
# Markdown + PDF report writer
# ---------------------------
def write_markdown_report(results: Dict, acc_lat_path: str, thr_path: str, out_md: str):
    # write summary markdown with speedup table relative to PyTorch GPU baseline with default bs
    with open(out_md, "w") as f:
        f.write("# Part D — Deployment Benchmarks\n\n")
        f.write("## Results (accuracy, latency, throughput)\n\n")
        f.write("| Backend | Accuracy | Latency (ms/img) | Throughput (img/s) |\n")
        f.write("|---|---:|---:|---:|\n")
        for k, v in results.items():
            if isinstance(v, dict) and "accuracy" in v:
                f.write(f"| {k} | {v['accuracy']:.4f} | {v['latency_ms']:.2f} | {v['throughput_img_s']:.2f} |\n")
            else:
                f.write(f"| {k} | - | - | - |\n")
        f.write("\n## Plots\n\n")
        if acc_lat_path:
            f.write(f"![Accuracy vs Latency]({os.path.basename(acc_lat_path)})\n\n")
        if thr_path:
            f.write(f"![Throughput]({os.path.basename(thr_path)})\n\n")
        # speedup table vs GPU baseline
        base_key = None
        for k in results:
            if k.startswith("pytorch_gpu_bs"):
                base_key = k
                break
        if base_key and isinstance(results[base_key], dict) and results[base_key].get("throughput_img_s", 0) > 0:
            f.write("## Relative speedups (vs PyTorch GPU baseline)\n\n")
            f.write("| Backend | Speedup × |\n|---|---:|\n")
            base_thr = results[base_key]["throughput_img_s"]
            for k, v in results.items():
                if isinstance(v, dict) and v.get("throughput_img_s", 0) > 0:
                    f.write(f"| {k} | {v['throughput_img_s'] / base_thr:.2f}× |\n")
        f.write("\n## Observations\n")
        f.write("- FP16 tends to be faster than FP32 with little accuracy change on supported hardware.\n")
        f.write("- INT8 can yield large speedups; watch for accuracy drop depending on calibration.\n")
        f.write("- Larger batch sizes typically improve throughput and decrease per-image latency.\n")
    print(f"[OK] Markdown report written: {out_md}")

def create_pdf_report(md_path: str, images: list, pdf_path: str):
    # Simple PDF creation using matplotlib PdfPages: first page includes text from markdown (plain), then images pages
    with open(md_path, "r") as f:
        md_text = f.read()
    with PdfPages(pdf_path) as pdf:
        # First page: text (split into lines)
        fig = plt.figure(figsize=(8.27, 11.69))  # A4
        fig.text(0.01, 0.99, md_text, va='top', wrap=True, fontsize=8)
        pdf.savefig(fig); plt.close(fig)
        # Add each image as its own page
        for img in images:
            if img and os.path.exists(img):
                fig = plt.figure(figsize=(8.27, 11.69))
                ax = fig.add_subplot(111)
                ax.axis('off')
                img_arr = plt.imread(img)
                ax.imshow(img_arr)
                pdf.savefig(fig); plt.close(fig)
    print(f"[OK] PDF report written: {pdf_path}")

# ---------------------------
# Main function: runs the full pipeline
# ---------------------------
def run_pipeline(data_dir: str, checkpoint: str=None, input_size:int=224, batch_size:int=8, num_classes:int=6, calib_samples:int=200):
    results = {}
    tf_test = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    test_path = os.path.join(data_dir, "test")
    if not os.path.isdir(test_path):
        raise RuntimeError(f"Test directory not found: {test_path}")
    test_ds = datasets.ImageFolder(test_path, transform=tf_test)

    # Build model and load weights (if provided)
    model = FlexibleCNNSimple(input_size=input_size, num_classes=num_classes)
    if checkpoint and os.path.exists(checkpoint):
        ckpt = torch.load(checkpoint, map_location="cpu")
        # ckpt may be state_dict or full dict
        try:
            model.load_state_dict(ckpt)
        except Exception:
            # try common keys
            sd = ckpt.get("state_dict", ckpt.get("model_state", None)) if isinstance(ckpt, dict) else None
            if sd is not None:
                model.load_state_dict(sd)
            else:
                raise
        print(f"[OK] Loaded checkpoint: {checkpoint}")
    else:
        print("[WARN] No checkpoint loaded; using random weights.")

    # Export ONNX
    onnx_path = os.path.join(RESULTS_DIR, "model.onnx")
    export_onnx(model, input_size, onnx_path)

    # Batch sizes to evaluate
    batch_sizes = [1, batch_size, 32]
    # Ensure unique and positive
    batch_sizes = sorted(list({bs for bs in batch_sizes if bs > 0}))

    # Precompute calibration loader (train subset)
    calib_loader = None
    try:
        calib_loader = get_calibration_loader(data_dir, input_size, batch_size=min(batch_size, 16), num_samples=calib_samples)
        print(f"[OK] Calibration loader prepared with {len(calib_loader.dataset)} samples.")
    except Exception as e:
        print("[WARN] Calibration loader not prepared:", e)
        calib_loader = None

    # For each batch size, evaluate PyTorch and TRT if available
    for bs in batch_sizes:
        loader = DataLoader(test_ds, batch_size=bs, shuffle=False)
        key_cpu = f"pytorch_cpu_bs{bs}"
        results[key_cpu] = evaluate_pytorch(model, loader, torch.device("cpu"))
        print(f"[INFO] {key_cpu}: {results[key_cpu]}")
        if torch.cuda.is_available():
            key_gpu = f"pytorch_gpu_bs{bs}"
            results[key_gpu] = evaluate_pytorch(model, loader, torch.device("cuda"))
            print(f"[INFO] {key_gpu}: {results[key_gpu]}")

        # TensorRT builds & eval
        if TRT_AVAILABLE:
            # FP32
            try:
                eng_fp32 = build_trt_engine_from_onnx(onnx_path, fp16=False, int8=False)
                key = f"trt_fp32_bs{bs}"
                results[key] = evaluate_trt_engine(eng_fp32, loader)
                print(f"[INFO] {key}: {results[key]}")
            except Exception as e:
                results[f"trt_fp32_bs{bs}"] = {"error": str(e)}
                print(f"[WARN] trt_fp32_bs{bs} failed: {e}")

            # FP16
            try:
                eng_fp16 = build_trt_engine_from_onnx(onnx_path, fp16=True, int8=False)
                key = f"trt_fp16_bs{bs}"
                results[key] = evaluate_trt_engine(eng_fp16, loader)
                print(f"[INFO] {key}: {results[key]}")
            except Exception as e:
                results[f"trt_fp16_bs{bs}"] = {"error": str(e)}
                print(f"[WARN] trt_fp16_bs{bs} failed: {e}")

            # INT8 (needs calibrator)
            try:
                if calib_loader is None:
                    raise RuntimeError("Calibration loader not available for INT8")
                calibrator = EntropyCalibrator(calib_loader, cache_file=os.path.join(RESULTS_DIR, f"calib_bs{bs}.bin"), max_batches=math.ceil(calib_samples / calib_loader.batch_size))
                eng_int8 = build_trt_engine_from_onnx(onnx_path, fp16=False, int8=True, calibrator=calibrator)
                key = f"trt_int8_bs{bs}"
                results[key] = evaluate_trt_engine(eng_int8, loader)
                print(f"[INFO] {key}: {results[key]}")
            except Exception as e:
                results[f"trt_int8_bs{bs}"] = {"error": str(e)}
                print(f"[WARN] trt_int8_bs{bs} failed: {e}")
        else:
            # TRT not available — mark keys
            results[f"trt_fp32_bs{bs}"] = {"error": "tensorrt_not_available"}
            results[f"trt_fp16_bs{bs}"] = {"error": "tensorrt_not_available"}
            results[f"trt_int8_bs{bs}"] = {"error": "tensorrt_not_available"}

    # Save raw results
    json_path = os.path.join(RESULTS_DIR, "benchmarks.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[OK] Saved results JSON -> {json_path}")

    # Plots
    acc_lat_path = os.path.join(RESULTS_DIR, "comparison.png")
    thr_path = os.path.join(RESULTS_DIR, "throughput.png")
    plot_accuracy_latency(results, acc_lat_path)
    plot_throughput(results, thr_path)

    # Markdown and PDF reports
    md_path = os.path.join(RESULTS_DIR, "README_PartD.md")
    write_markdown_report(results, acc_lat_path, thr_path, md_path)

    pdf_path = os.path.join(RESULTS_DIR, "PartD_Report.pdf")
    images = [acc_lat_path, thr_path]
    create_pdf_report(md_path, images, pdf_path)

    return results

# ---------------------------
# Dual-mode execution (Notebook vs CLI)
# ---------------------------
if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.argv[0] or "pytest" in sys.argv[0]:
        # Notebook / interactive defaults
        class Args:
            data_dir = DEFAULT_DATA_DIR
            checkpoint = DEFAULT_CHECKPOINT
            input_size = 224
            batch_size = 8
            num_classes = 6
            calib_samples = 200
        args = Args()
        run_pipeline(args.data_dir, checkpoint=args.checkpoint, input_size=args.input_size,
                     batch_size=args.batch_size, num_classes=args.num_classes, calib_samples=args.calib_samples)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--data_dir", type=str, required=True)
        parser.add_argument("--checkpoint", type=str, required=False, default=DEFAULT_CHECKPOINT)
        parser.add_argument("--input_size", type=int, default=224)
        parser.add_argument("--batch_size", type=int, default=8)
        parser.add_argument("--num_classes", type=int, default=6)
        parser.add_argument("--calib_samples", type=int, default=200)
        parsed = parser.parse_args()
        run_pipeline(parsed.data_dir, checkpoint=parsed.checkpoint, input_size=parsed.input_size,
                     batch_size=parsed.batch_size, num_classes=parsed.num_classes, calib_samples=parsed.calib_samples)


[WARN] No checkpoint loaded; using random weights.
[OK] ONNX exported to: partD_results/model.onnx
[OK] Calibration loader prepared with 200 samples.


/tmp/ipykernel_2249115/1555964531.py:97: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


[INFO] pytorch_cpu_bs1: {'accuracy': 0.19094488188976377, 'latency_ms': 14.41360914375721, 'throughput_img_s': 69.3788758961261, 'batch_size': 1}
[INFO] pytorch_gpu_bs1: {'accuracy': 0.19094488188976377, 'latency_ms': 1.3396443375642961, 'throughput_img_s': 746.4667837272183, 'batch_size': 1}
[WARN] trt_fp32_bs1 failed: 'tensorrt_bindings.tensorrt.IBuilderConfig' object has no attribute 'max_workspace_size'
[WARN] trt_fp16_bs1 failed: 'tensorrt_bindings.tensorrt.IBuilderConfig' object has no attribute 'max_workspace_size'
[WARN] trt_int8_bs1 failed: 'tensorrt_bindings.tensorrt.IBuilderConfig' object has no attribute 'max_workspace_size'
[INFO] pytorch_cpu_bs8: {'accuracy': 0.19094488188976377, 'latency_ms': 15.350439582107356, 'throughput_img_s': 517.0861692620216, 'batch_size': 8}
[INFO] pytorch_gpu_bs8: {'accuracy': 0.19094488188976377, 'latency_ms': 0.2519444224162726, 'throughput_img_s': 31504.96416580855, 'batch_size': 8}
[WARN] trt_fp32_bs8 failed: 'tensorrt_bindings.tensorrt.IBu

In [13]:
import os, time, json, math, shutil, sys, io
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report

try:
    import tensorrt as trt; TRT_AVAILABLE=True
except Exception:
    trt=None; TRT_AVAILABLE=False
try:
    import pycuda.driver as cuda; import pycuda.autoinit; PYCUDA_AVAILABLE=True
except Exception:
    cuda=None; PYCUDA_AVAILABLE=False
try:
    import psutil; PSUTIL_AVAILABLE=True
except Exception:
    psutil=None; PSUTIL_AVAILABLE=False
try:
    import wandb; WANDB_AVAILABLE=True
except Exception:
    wandb=None; WANDB_AVAILABLE=False

DATA_DIR = "/home/nagaraj/Garbage_classification_files/Garbage classification"
CHECKPOINT = "partB_results/best_model.pth"
RESULTS_DIR = "partD_results"
PROJECT = "Atri"
ENTITY = "cs24s023-iitm-ac-in"
os.makedirs(RESULTS_DIR, exist_ok=True)

class FlexibleCNNSimple(nn.Module):
    def __init__(self, input_size=224, num_classes=6, filters=(32,64,128,256,512),
                 kernel_sizes=(3,3,3,3,3), activation="relu", dense_neurons=512,
                 use_bn=True, dropout=0.3):
        super().__init__()
        act = {"relu": nn.ReLU(), "gelu": nn.GELU(), "silu": nn.SiLU(), "mish": nn.Mish()}.get(activation, nn.ReLU())
        layers=[]; in_ch=3
        for i, out_ch in enumerate(filters):
            k = kernel_sizes[i] if i < len(kernel_sizes) else 3
            layers.append(nn.Conv2d(in_ch, out_ch, k, padding=k//2))
            if use_bn: layers.append(nn.BatchNorm2d(out_ch))
            layers.append(act); layers.append(nn.MaxPool2d(2)); in_ch=out_ch
        self.conv=nn.Sequential(*layers)
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.fc1=nn.Linear(filters[-1], dense_neurons)
        self.act=act
        self.drop=nn.Dropout(dropout) if dropout>0 else None
        self.fc2=nn.Linear(dense_neurons,num_classes)

    def forward(self,x):
        x=self.conv(x); x=self.pool(x); x=torch.flatten(x,1)
        x=self.fc1(x); x=self.act(x)
        if self.drop: x=self.drop(x)
        return self.fc2(x)

def export_onnx(model, input_size, onnx_path):
    model.eval()
    dummy=torch.randn(1,3,input_size,input_size)
    try:
        torch.onnx.export(model, dummy, onnx_path, opset_version=18,
            input_names=["input"], output_names=["output"],
            dynamic_axes={"input":{0:"batch"},"output":{0:"batch"}})
        print("[INFO] Exported ONNX ->", onnx_path)
    except Exception as e:
        print("[WARN] ONNX export issue:", e, "trying simpler export")
        try:
            torch.onnx.export(model, dummy, onnx_path, opset_version=13,
                input_names=["input"], output_names=["output"],
                dynamic_axes={"input":{0:"batch"},"output":{0:"batch"}})
            print("[INFO] Exported ONNX (legacy) ->", onnx_path)
        except Exception as e2:
            raise RuntimeError("ONNX export failed: " + str(e2))

def make_test_loader(data_dir,input_size=224,batch_size=16,eval_samples=500):
    test_dir=os.path.join(data_dir,"test"); assert os.path.isdir(test_dir), f"Test dir missing: {test_dir}"
    norm=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    tf=transforms.Compose([transforms.Resize((input_size,input_size)),transforms.ToTensor(),norm])
    ds=datasets.ImageFolder(test_dir,transform=tf)
    if eval_samples>0 and len(ds)>eval_samples:
        idx=np.random.choice(len(ds),eval_samples,replace=False).tolist(); ds=Subset(ds,idx)
    loader=DataLoader(ds,batch_size=batch_size,shuffle=False,pin_memory=True)
    classes=ds.dataset.classes if isinstance(ds,Subset) else ds.classes
    return loader,classes

def get_calibration_loader(data_dir,input_size=224,batch_size=8,num_samples=200):
    train_dir=os.path.join(data_dir,"train");
    if not os.path.isdir(train_dir): return None
    norm=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    tf=transforms.Compose([transforms.Resize((input_size,input_size)),transforms.ToTensor(),norm])
    ds=datasets.ImageFolder(train_dir,transform=tf)
    n=min(len(ds),num_samples);
    if n==0: return None
    idx=np.random.choice(len(ds),n,replace=False).tolist()
    return DataLoader(Subset(ds,idx),batch_size=min(batch_size,n),shuffle=True,pin_memory=True)

def plot_confusion(cm,classes,out_path,title="Confusion"):
    fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cm,cmap="Blues")
    ax.set_title(title); plt.colorbar(im,ax=ax)
    ax.set_xticks(np.arange(len(classes))); ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes,rotation=45,ha="right"); ax.set_yticklabels(classes)
    for i in range(len(classes)):
        for j in range(len(classes)): ax.text(j,i,int(cm[i][j]),ha="center",va="center",color="black")
    plt.tight_layout(); fig.savefig(out_path); plt.close(fig)

def visualize_misclassified(imgs,labels,preds,classes,out_path,max_samples=30):
    imgs_np=imgs.numpy(); labs=labels.numpy().tolist(); pres=preds.numpy().tolist()
    wrong=[(i,labs[i],pres[i]) for i in range(len(labs)) if labs[i]!=pres[i]]
    wrong=wrong[:max_samples]; if_len = len(wrong)
    if if_len==0:
        fig=plt.figure(); plt.text(0.5,0.5,"No misclassifications",ha="center",va="center"); plt.axis("off")
        fig.savefig(out_path); plt.close(fig); return
    rows=math.ceil(len(wrong)/3); cols=3
    fig,axes=plt.subplots(rows,cols,figsize=(cols*3,rows*3)); axes=axes.flatten()
    for idx,(i,t,p) in enumerate(wrong):
        img=imgs_np[i].transpose(1,2,0); img=(img*[0.229,0.224,0.225]+[0.485,0.456,0.406]).clip(0,1)
        axes[idx].imshow(img); axes[idx].set_title(f"T:{classes[t]} P:{classes[p]}",fontsize=8); axes[idx].axis("off")
    for j in range(len(wrong),len(axes)): axes[j].axis("off")
    plt.tight_layout(); plt.savefig(out_path); plt.close(fig)

def evaluate_pytorch(model,loader,device,classes):
    model.to(device).eval(); correct=0; total=0; times=[]; preds_all=[]; labels_all=[]
    imgs_all=[]; labs_all=[]; pres_all=[]
    with torch.no_grad():
        for imgs,labels in loader:
            imgs_dev,labels_dev=imgs.to(device),labels.to(device)
            t0=time.perf_counter(); out=model(imgs_dev); t1=time.perf_counter()
            preds=out.argmax(1); correct+=(preds==labels_dev).sum().item(); total+=labels_dev.size(0)
            times.append((t1-t0)/imgs.size(0))
            preds_all+=preds.cpu().tolist(); labels_all+=labels.cpu().tolist()
            if sum([x.size(0) for x in imgs_all]) < 30:
                imgs_all.append(imgs.cpu()); labs_all.append(labels.cpu()); pres_all.append(preds.cpu())
    acc=correct/total if total>0 else 0
    cm=confusion_matrix(labels_all,preds_all,labels=list(range(len(classes))))
    rep=classification_report(labels_all,preds_all,target_names=classes,output_dict=True,zero_division=0)
    first_imgs=torch.cat(imgs_all) if imgs_all else None
    first_labels=torch.cat(labs_all) if labs_all else None
    first_preds=torch.cat(pres_all) if pres_all else None
    return {"accuracy":acc,"latency_ms":np.mean(times)*1000 if times else None,"throughput_img_s":(total/sum(times)) if times and sum(times)>0 else None,
            "cm":cm.tolist(),"report":rep,"first_imgs":first_imgs,
            "first_labels":first_labels,"first_preds":first_preds}


if TRT_AVAILABLE:
    TRT_LOGGER=trt.Logger(trt.Logger.WARNING)
    class EntropyCalibrator(trt.IInt8EntropyCalibrator2):
        def __init__(self,loader,cache_file="calib.bin",max_batches=10):
            super().__init__(); self.loader=loader; self.iterator=iter(loader)
            self.cache_file=cache_file; self.max_batches=max_batches; self.current=0; self.device_input=None
        def get_batch_size(self): return self.loader.batch_size
        def get_batch(self,names):
            if self.current>=self.max_batches: return None
            try: imgs,_=next(self.iterator)
            except StopIteration: return None
            arr=imgs.numpy().astype(np.float32)
            if self.device_input is None: self.device_input=cuda.mem_alloc(arr.nbytes)
            cuda.memcpy_htod(self.device_input,arr); self.current+=1
            return [int(self.device_input)]
        def read_calibration_cache(self):
            return open(self.cache_file,"rb").read() if os.path.exists(self.cache_file) else None
        def write_calibration_cache(self,cache): open(self.cache_file,"wb").write(cache)

    def build_trt_engine(onnx_path,fp16=False,int8=False,calibrator=None,workspace=(1<<30)):
        builder=trt.Builder(TRT_LOGGER); net=builder.create_network(1<<int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
        parser=trt.OnnxParser(net,TRT_LOGGER)
        with open(onnx_path,"rb") as f:
            if not parser.parse(f.read()): raise RuntimeError("ONNX parse failed")
        config=builder.create_builder_config()
        try: config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE,workspace)
        except: pass
        if fp16: config.set_flag(trt.BuilderFlag.FP16)
        if int8:
            config.set_flag(trt.BuilderFlag.INT8)
            if calibrator is None: raise RuntimeError("Need calibrator for INT8")
            config.int8_calibrator=calibrator
        engine=builder.build_engine(net,config)
        if engine is None: raise RuntimeError("TRT build failed")
        return engine

    def evaluate_trt(engine,loader,classes):
        if not PYCUDA_AVAILABLE: return {"error":"pycuda unavailable"}
        ctx=engine.create_execution_context()
      
        in_idx = None; out_idx = None
        for i in range(engine.num_bindings):
            if engine.binding_is_input(i): in_idx = i
            else: out_idx = i
        in_shape = engine.get_binding_shape(in_idx); out_shape = engine.get_binding_shape(out_idx)
   
        d_input = cuda.mem_alloc(int(trt.volume(in_shape)*4)); d_output = cuda.mem_alloc(int(trt.volume(out_shape)*4))
        stream = cuda.Stream()
        correct=0; total=0; times=[]; preds_all=[]; labels_all=[]
        imgs_all=[]; labs_all=[]; pres_all=[]
        for imgs,labels in loader:
            arr = imgs.numpy().astype(np.float32)
            cuda.memcpy_htod_async(d_input, arr, stream)
            t0=time.perf_counter(); ctx.execute_v2([int(d_input), int(d_output)]); stream.synchronize(); t1=time.perf_counter()
            out = np.empty(int(trt.volume(out_shape)), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream); stream.synchronize()
        
            batch = imgs.shape[0]
            out = out.reshape((batch, -1))
            preds = out.argmax(1)
            correct += (preds == labels.numpy()).sum()
            total += labels.size(0)
            times.append((t1-t0)/imgs.size(0))
            preds_all += preds.tolist(); labels_all += labels.tolist()
            if sum([x.shape[0] for x in imgs_all]) < 30:
                imgs_all.append(imgs); labs_all.append(labels); pres_all.append(torch.from_numpy(preds))
        acc = (correct/total) if total>0 else 0
        cm = confusion_matrix(labels_all,preds_all,labels=list(range(len(classes))))
        rep = classification_report(labels_all,preds_all,target_names=classes,output_dict=True,zero_division=0)
        return {"accuracy":float(acc),"latency_ms":np.mean(times)*1000 if times else None,"throughput_img_s":(total/sum(times)) if times and sum(times)>0 else None,
                "cm":cm.tolist(),"report":rep,"first_imgs":torch.cat(imgs_all) if imgs_all else None,
                "first_labels":torch.cat(labs_all) if labs_all else None,"first_preds":torch.cat(pres_all) if pres_all else None}

def plot_speed_comparison(results, out_path):
    records=[]
    for k,v in results.items():
        if not isinstance(v,dict): continue
        rec={"backend":k,"accuracy":v.get("accuracy"),"latency_ms":v.get("latency_ms"),"throughput":v.get("throughput_img_s")}
        records.append(rec)
    df=pd.DataFrame(records)
    if df.empty:
        print("[WARN] No numeric results to plot for speed comparison")
        return
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    df_plot=df.dropna(subset=["latency_ms"])
    ax[0].bar(df_plot["backend"], df_plot["latency_ms"])
    ax[0].set_title("Latency per image (ms)")
    ax[0].set_ylabel("ms")
    df_th=df.dropna(subset=["throughput"])
    ax[1].bar(df_th["backend"], df_th["throughput"])
    ax[1].set_title("Throughput (images / s)")
    ax[1].set_ylabel("img/s")
    plt.xticks(rotation=45,ha="right")
    plt.tight_layout()
    fig.savefig(out_path); plt.close(fig)

def generate_markdown_report(results, out_dir):
    md_path = os.path.join(out_dir,"report.md")
    pdf_path = os.path.join(out_dir,"report.pdf")
    # Basic analysis text
    def safe_stat(k):
        v = results.get(k, {})
        return v.get("accuracy"), v.get("latency_ms"), v.get("throughput_img_s")
    lines=[]
    lines.append("# Inference Benchmark Report\n")
    lines.append(f"Generated: {time.asctime()}\n")
    lines.append("## Overview\n")
    lines.append("This report summarizes inference accuracy, latency and throughput across different backends (PyTorch CPU, PyTorch GPU, TensorRT FP32/FP16/INT8 where available).\n")
    lines.append("## Results Summary\n")
    for k in ["pytorch_cpu","pytorch_gpu","trt_fp32","trt_fp16","trt_int8"]:
        if k in results:
            v=results[k]
            lines.append(f"### {k}\n")
            if "error" in v:
                lines.append(f"- Error: `{v['error']}`\n")
            else:
                acc = v.get("accuracy"); lat=v.get("latency_ms"); thr=v.get("throughput_img_s")
                lines.append(f"- Accuracy: **{acc:.4f}**\n" if isinstance(acc,(int,float)) else f"- Accuracy: {acc}\n")
                lines.append(f"- Latency per image (ms): {lat:.2f}\n" if lat else "- Latency per image (ms): N/A\n")
                lines.append(f"- Throughput (img/s): {thr:.2f}\n" if thr else "- Throughput (img/s): N/A\n")
    lines.append("## Automated Observations\n")
    cpu = results.get("pytorch_cpu",{})
    gpu = results.get("pytorch_gpu",{})
    trt_fp32 = results.get("trt_fp32",{})
    try:
        if cpu.get("latency_ms") and gpu.get("latency_ms"):
            speedup = cpu["latency_ms"]/gpu["latency_ms"]
            lines.append(f"- GPU vs CPU speedup (latency): ~{speedup:.2f}x\n")
        if cpu.get("latency_ms") and trt_fp32.get("latency_ms"):
            speedup_trt = cpu["latency_ms"]/trt_fp32["latency_ms"]
            lines.append(f"- TensorRT FP32 vs CPU speedup (latency): ~{speedup_trt:.2f}x\n")
    except Exception:
        pass
    lines.append("\n## Plots and Confusion Matrices\n")
    lines.append("- `cm_cpu.png`, `cm_gpu.png`, `cm_trt_fp32.png`, etc. contain confusion matrices.\n")
    lines.append("- `mis_cpu.png`, `mis_gpu.png`, `mis_trt_fp32.png`, etc. contain example misclassified images.\n")
    with open(md_path,"w") as f:
        f.write("\n".join(lines))
    with PdfPages(pdf_path) as pdf:
      
        fig = plt.figure(figsize=(8.27, 11.69))  # A4
        fig.text(0.01, 0.99, "\n".join(lines[:50]), va="top", wrap=True, fontsize=8)
        pdf.savefig(fig); plt.close(fig)
     
        for fname in sorted(os.listdir(out_dir)):
            if fname.lower().endswith(".png"):
                try:
                    fig = plt.figure()
                    img = plt.imread(os.path.join(out_dir,fname))
                    plt.imshow(img); plt.axis("off")
                    pdf.savefig(fig); plt.close(fig)
                except Exception:
                    pass
    print(f"[INFO] Markdown report: {md_path} | PDF report: {pdf_path}")
    return md_path, pdf_path

def maybe_log_wandb(results, out_dir, project=PROJECT, entity=ENTITY):
    if not WANDB_AVAILABLE:
        print("[INFO] wandb not available; skipping W&B logging")
        return None
    try:
        wandb.init(project=project, entity=entity, config={"run":"partD_benchmark"})
        # log scalar metrics
        for k,v in results.items():
            if not isinstance(v,dict): continue
            summary = {}
            if "accuracy" in v: summary[f"{k}/accuracy"] = v["accuracy"]
            if "latency_ms" in v: summary[f"{k}/latency_ms"] = v["latency_ms"]
            if "throughput_img_s" in v: summary[f"{k}/throughput"] = v["throughput_img_s"]
            wandb.log(summary)
    
        for fname in os.listdir(out_dir):
            if fname.lower().endswith(".png"):
                wandb.log({f"image/{fname}": wandb.Image(os.path.join(out_dir,fname))})
        wandb.finish()
        print("[INFO] Logged results to W&B")
    except Exception as e:
        print("[WARN] W&B logging failed:", e)


def run_pipeline(data_dir=DATA_DIR,checkpoint=CHECKPOINT,input_size=224,batch_size=16,
                 num_classes=6,calib_samples=200,eval_samples=500,use_wandb=True,zip_results=True):
   
    test_loader,classes=make_test_loader(data_dir,input_size,batch_size,eval_samples)
    calib_loader=get_calibration_loader(data_dir,input_size,batch_size,num_samples=calib_samples)
    model=FlexibleCNNSimple(input_size,num_classes)
    if checkpoint and os.path.exists(checkpoint):
        ck=torch.load(checkpoint,map_location="cpu")
        try:
            model.load_state_dict(ck)
        except Exception:
            model.load_state_dict(ck.get("state_dict",ck))
 
    onnx_path=os.path.join(RESULTS_DIR,"model.onnx"); export_onnx(model,input_size,onnx_path)

    results={}

    print("[INFO] Evaluating PyTorch CPU...")
    r_cpu=evaluate_pytorch(model,test_loader,torch.device("cpu"),classes); results["pytorch_cpu"]=r_cpu
    plot_confusion(np.array(r_cpu["cm"]),classes,os.path.join(RESULTS_DIR,"cm_cpu.png"),title="Confusion (PyTorch CPU)")
    if r_cpu.get("first_imgs") is not None:
        visualize_misclassified(r_cpu["first_imgs"],r_cpu["first_labels"],r_cpu["first_preds"],classes,os.path.join(RESULTS_DIR,"mis_cpu.png"))

 
    if torch.cuda.is_available():
        print("[INFO] Evaluating PyTorch GPU...")
        r_gpu=evaluate_pytorch(model,test_loader,torch.device("cuda"),classes); results["pytorch_gpu"]=r_gpu
        plot_confusion(np.array(r_gpu["cm"]),classes,os.path.join(RESULTS_DIR,"cm_gpu.png"),title="Confusion (PyTorch GPU)")
        if r_gpu.get("first_imgs") is not None:
            visualize_misclassified(r_gpu["first_imgs"],r_gpu["first_labels"],r_gpu["first_preds"],classes,os.path.join(RESULTS_DIR,"mis_gpu.png"))
    else:
        print("[INFO] CUDA not available; skipping PyTorch GPU evaluation")

   
    if TRT_AVAILABLE:
        print("[INFO] Building/evaluating TensorRT engines...")
        try:
            r_trt_fp32 = build_trt_engine(onnx_path,fp16=False,int8=False)
            res = evaluate_trt(r_trt_fp32,test_loader,classes)
            results["trt_fp32"]=res
            if "cm" in res:
                plot_confusion(np.array(res["cm"]),classes,os.path.join(RESULTS_DIR,"cm_trt_fp32.png"),title="Confusion (TRT FP32)")
            if res.get("first_imgs") is not None:
                visualize_misclassified(res["first_imgs"],res["first_labels"],res["first_preds"],classes,os.path.join(RESULTS_DIR,"mis_trt_fp32.png"))
        except Exception as e:
            results["trt_fp32"]={"error":str(e)}
        try:
            r_trt_fp16 = build_trt_engine(onnx_path,fp16=True,int8=False)
            res = evaluate_trt(r_trt_fp16,test_loader,classes)
            results["trt_fp16"]=res
            if isinstance(res,dict) and "cm" in res:
                plot_confusion(np.array(res["cm"]),classes,os.path.join(RESULTS_DIR,"cm_trt_fp16.png"),title="Confusion (TRT FP16)")
                if res.get("first_imgs") is not None:
                    visualize_misclassified(res["first_imgs"],res["first_labels"],res["first_preds"],classes,os.path.join(RESULTS_DIR,"mis_trt_fp16.png"))
        except Exception as e:
            results["trt_fp16"]={"error":str(e)}
        try:
            if calib_loader:
                calib=EntropyCalibrator(calib_loader)
                r_trt_int8 = build_trt_engine(onnx_path,fp16=False,int8=True,calibrator=calib)
                res = evaluate_trt(r_trt_int8,test_loader,classes)
                results["trt_int8"]=res
                if isinstance(res,dict) and "cm" in res:
                    plot_confusion(np.array(res["cm"]),classes,os.path.join(RESULTS_DIR,"cm_trt_int8.png"),title="Confusion (TRT INT8)")
                    if res.get("first_imgs") is not None:
                        visualize_misclassified(res["first_imgs"],res["first_labels"],res["first_preds"],classes,os.path.join(RESULTS_DIR,"mis_trt_int8.png"))
            else:
                results["trt_int8"]={"error":"no calib loader"}
        except Exception as e:
            results["trt_int8"]={"error":str(e)}
    else:
        print("[INFO] TensorRT not available; skipping TRT steps")

    def make_safe_for_json(v):
        if isinstance(v, dict):
            out={}
            for kk,vv in v.items():
                if isinstance(vv, (int,float,str,bool)) or vv is None:
                    out[kk]=vv
                elif isinstance(vv, list):
                    out[kk]=vv
                elif isinstance(vv, dict):
                    out[kk]=make_safe_for_json(vv)
                else:
                    # fallback: stringify small items
                    try:
                        out[kk]=float(vv) if isinstance(vv,(np.floating,float)) else str(type(vv))
                    except:
                        out[kk]=str(type(vv))
            return out
        else:
            return str(type(v))
    safe = {k: make_safe_for_json(v) for k,v in results.items()}
    with open(os.path.join(RESULTS_DIR,"benchmarks.json"),"w") as f: json.dump(safe,f,indent=2)
    print("[DONE] Results saved to",RESULTS_DIR)

  
    plot_speed_comparison(results, os.path.join(RESULTS_DIR,"speed_comparison.png"))

 
    md_path, pdf_path = generate_markdown_report(results, RESULTS_DIR)


    if use_wandb:
        maybe_log_wandb(results, RESULTS_DIR)


    if zip_results:
        shutil.make_archive(RESULTS_DIR, 'zip', RESULTS_DIR)
        print("[INFO] Zipped results:", RESULTS_DIR + ".zip")

    return results


if __name__=="__main__":
    if "ipykernel_launcher" in sys.argv[0]:
        run_pipeline()
    else:
        import argparse
        p=argparse.ArgumentParser()
        p.add_argument("--data_dir",type=str,default=DATA_DIR)
        p.add_argument("--checkpoint",type=str,default=CHECKPOINT)
        p.add_argument("--input_size",type=int,default=224)
        p.add_argument("--batch_size",type=int,default=16)
        p.add_argument("--use_wandb",action="store_true")
        p.add_argument("--no_zip",action="store_true")
        args=p.parse_args()
        run_pipeline(data_dir=args.data_dir,checkpoint=args.checkpoint,
                     input_size=args.input_size,batch_size=args.batch_size,
                     use_wandb=args.use_wandb,zip_results=(not args.no_zip))


/tmp/ipykernel_2249115/699884196.py:83: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(model, dummy, onnx_path, opset_version=18,


[INFO] Exported ONNX -> partD_results/model.onnx
[INFO] Evaluating PyTorch CPU...
[INFO] Evaluating PyTorch GPU...
[INFO] Building/evaluating TensorRT engines...


/tmp/ipykernel_2249115/699884196.py:202: DeprecationWarning: Use Deprecated in TensorRT 10.1. Superseded by explicit quantization. instead.
  config.int8_calibrator=calibrator


[DONE] Results saved to partD_results
[INFO] Markdown report: partD_results/report.md | PDF report: partD_results/report.pdf


pytorch_cpu/accuracy,▁
pytorch_cpu/latency_ms,▁
pytorch_cpu/throughput,▁
pytorch_gpu/accuracy,▁
pytorch_gpu/latency_ms,▁
pytorch_gpu/throughput,▁
pytorch_cpu/accuracy,0.056
pytorch_cpu/latency_ms,14.4489
pytorch_cpu/throughput,1081.39741
pytorch_gpu/accuracy,0.056
pytorch_gpu/latency_ms,0.11069


[INFO] Logged results to W&B
[INFO] Zipped results: partD_results.zip
